# Generator API Design

## Proposed Function

```python
def generate(
    data: str,
    output_path: str,
    n_bits: int = 8,
    key: bytes | None = None
) -> dict:
    """
    Create a tamper-evident QR for `data`, write the QR image to
    `output_path`, and return metadata describing what was generated.
    """

In [1]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath('..'))

In [2]:
from quantum_qr.generator import generate

result = generate(
    data="pay alice $10",
    output_path="../data/alice_payment.png"
)

print(result["payload"])

Counts: {'01011111010111110000000010101101000000001100000100001101110000001100101110101010110011001110011110110101101001100110010111011000': 1}

128 Random Bits (List):
[0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0]

128 Random Bits (Hexadecimal):
1ba665ade73355d303b08300b500fafa
{'version': '1', 'data': 'pay alice $10', 'nonce': '1ba665ade73355d303b08300b500fafa', 'tag': '01001011'}


In [4]:
result2 = generate(
    data="pay alice $10",
    output_path="../data/alice_payment_2.png"
)

print(result2["payload"])

Counts: {'01100111101100110000111011101001001010011101111111011001111101110101101011000100110010011101010010111110101010110010001011100100': 1}

128 Random Bits (List):
[0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0]

128 Random Bits (Hexadecimal):
2744d57d2b93235aef9bfb949770cde6
{'version': '1', 'data': 'pay alice $10', 'nonce': '2744d57d2b93235aef9bfb949770cde6', 'tag': '01110101'}


In [5]:
# compare the two payloads
print("First nonce :", result["payload"]["nonce"])
print("Second nonce:", result2["payload"]["nonce"])

print()

print("First tag :", result["payload"]["tag"])
print("Second tag:", result2["payload"]["tag"])

First nonce : 1ba665ade73355d303b08300b500fafa
Second nonce: 2744d57d2b93235aef9bfb949770cde6

First tag : 01001011
Second tag: 01110101


In [6]:
print(
    result["qr_string"] ==
    result2["qr_string"]
)

False


# Observation: Quantum Random Nonces

I generated two QR codes using the same payload:

    "pay alice $10"

Even though the message was identical, the generated QR codes were different.

This happened because each QR contains a freshly generated 128-bit quantum nonce.

Since the nonce changes:

- the HMAC tag changes
- the payload changes
- the encoded QR string changes
- the QR image changes

This is a desirable security property because it prevents identical messages from producing identical QR codes.

Without a nonce, an attacker could easily recognize and replay previously captured QR codes.

The quantum-generated nonce ensures that every generated QR is unique, even when the underlying message remains the same.

1. Read the QR Back

In [10]:
from quantum_qr.qr_io import read_qr_code

qr_string = read_qr_code("../data/alice_payment.png")

print(qr_string)

eyJ2ZXJzaW9uIjoiMSIsImRhdGEiOiJwYXkgYWxpY2UgJDEwIiwibm9uY2UiOiIxYmE2NjVhZGU3MzM1NWQzMDNiMDgzMDBiNTAwZmFmYSIsInRhZyI6IjAxMDAxMDExIn0=


2. Decode the Payload

In [11]:
from quantum_qr.payload import decode_payload

decoded_payload = decode_payload(qr_string)

print(decoded_payload)

{'version': '1', 'data': 'pay alice $10', 'nonce': '1ba665ade73355d303b08300b500fafa', 'tag': '01001011'}


3. Compare with Original Payload

In [12]:
print("Generated Payload:")
print(result["payload"])

print("\nDecoded Payload:")
print(decoded_payload)

Generated Payload:
{'version': '1', 'data': 'pay alice $10', 'nonce': '1ba665ade73355d303b08300b500fafa', 'tag': '01001011'}

Decoded Payload:
{'version': '1', 'data': 'pay alice $10', 'nonce': '1ba665ade73355d303b08300b500fafa', 'tag': '01001011'}


4. Verify All Fields Match

In [13]:
assert decoded_payload == result["payload"]

print("✓ Payload matches exactly")

✓ Payload matches exactly


5. Recompute Expected Tag

In [14]:
from quantum_qr.config import get_key
from quantum_qr.payload import compute_tag

key = get_key()

tag_expected = compute_tag(
    key,
    decoded_payload["data"],
    decoded_payload["nonce"],
    n_bits=8
)

print("Observed Tag:", decoded_payload["tag"])
print("Expected Tag:", tag_expected)

Observed Tag: 01001011
Expected Tag: 01001011


6. Compute Secret

In [15]:
from quantum_qr.payload import tags_to_secret

secret = tags_to_secret(
    decoded_payload["tag"],
    tag_expected
)

print("Secret:", secret)

Secret: 00000000


In [16]:
assert secret == "00000000"

print("✓ Verification successful")

✓ Verification successful


# End-to-End Verification Test

Today I verified that a generated QR code can be successfully read and validated.

Verification steps:

1. Read the QR image using OpenCV.
2. Decode the Base64 payload.
3. Parse the JSON fields.
4. Recompute the HMAC-derived tag using the shared key.
5. Compare the observed tag against the expected tag using XOR.

The resulting secret string was:

    00000000

This indicates that:

    observed_tag == expected_tag

and therefore the QR has not been modified.

This is the exact condition that will later produce a constant oracle in the Deutsch–Jozsa / Bernstein–Vazirani verification stage.

Current verification result:

    secret = 00000000
    → authentic

Future quantum interpretation:

    secret = 00000000
    → constant oracle
    → measurement = 00000000

This experiment confirms that the QR generation and payload verification pipeline works correctly before introducing the quantum verification layer.